# Train a model from a config

This notebook trains one experiment using the TOML config. Before running it, edit the config to use a small model and a short training budget for a quick check. Each run gets its own folder containing the copied config, checkpoints, samples, and logs.

Paths are resolved from `.env` relative to the repository root.

In [ ]:
from radio_diffusion.paths import load_env

# Apply compiler and other machine settings before importing PyTorch.
load_env()


In [ ]:
from pathlib import Path

from radio_diffusion.paths import REPOSITORY_ROOT, get_path

from radio_diffusion.experiment import (
    build_data_module,
    build_trainer,
    build_training_module,
    load_config,
    seed_experiment,
)

## Choose the experiment config

Edit this TOML file before training if you want a smaller model or fewer epochs. The config copy saved in the run folder will preserve the exact settings used for that run.

In [ ]:
config_path = REPOSITORY_ROOT / "configs" / "experiment.toml"
config = load_config(config_path)

print("Experiment:", config["run"]["name"])
print("Output root:", get_path("OUTPUT_ROOT"))
print("Training settings:", config["trainer"])
print("Model settings:", config["model"])

## Prepare the data and training run

Dataset paths come from `.env` and the configured image size, unless the config supplies an explicit path. `prepare_data()` reuses processed data or preprocesses available raw FITS/UV files. Missing raw files raise an error with the manual sync command.

In [ ]:
seed_experiment(config)
data = build_data_module(config)
data.prepare_data()

training_module = build_training_module(config)
trainer = build_trainer(config)

checkpoint_callback = trainer.checkpoint_callback
run_dir = Path(checkpoint_callback.dirpath).parent

print("This run will be saved to:", run_dir)
print("Config copy:", run_dir / "config.toml")

## Train

The Trainer saves the lowest-`val_loss` checkpoint as `checkpoints/best.ckpt` and the latest checkpoint as `checkpoints/last.ckpt`. Sample grids are saved at the configured epoch interval.

In [ ]:
trainer.fit(training_module, datamodule=data)

## Review what was saved

In [ ]:
print("Run folder:", run_dir)
print("Best checkpoint:", checkpoint_callback.best_model_path)
print("Last checkpoint:", checkpoint_callback.last_model_path)
print("Training logs:", trainer.logger.log_dir)
print("Saved samples:", run_dir / "samples")